# 🔀 Parallel Agent Execution in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Fan-Out/Fan-In Pattern** - How to launch multiple independent agents from a single `START` node and merge their outputs into one final state
2. **Parallel Research Agents** - Building three specialized agents (research, creative, technical) that run concurrently and get combined by a synthesis node
3. **Map-Reduce Pattern** - Applying map-reduce in LangGraph to summarize a list of documents and then combine the individual summaries
4. **Graph Fan-Out Edges** - Using multiple `add_edge` calls from the same source node to trigger parallel branch execution

## Prerequisites
- Basic understanding of LangGraph `StateGraph`, nodes, and edges
- Familiarity with `TypedDict` state schemas
- An `OPENAI_API_KEY` set in your `.env` file

---
## 📦 Part 0: Environment Setup

We load environment variables and initialize the LLM shared by every agent node in this notebook. All agents below reuse this single `llm` instance with different system prompts to specialize their behavior.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports, API Keys, and LLM Initialization
# ============================================================================
import asyncio

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

# Load API keys from .env (requires OPENAI_API_KEY)
load_dotenv()

# Initialize the LLM used by every agent node in this notebook
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

print(f"✅ Environment loaded and LLM initialized: {llm.model_name}")

---
## 🔀 Part 1: Fan-Out / Fan-In — Parallel Research Agents

The **fan-out/fan-in** pattern starts several independent agent nodes from the same point in the graph, lets them run concurrently, then converges their results into a single synthesis step. This is useful whenever a task benefits from multiple independent perspectives that can be gathered without waiting on each other.

### Key Concepts:
- **Fan-out**: Multiple `add_edge(START, node)` calls send execution to several nodes at once
- **Fan-in**: Multiple nodes each add an edge into the same downstream node, which only runs once all of its inputs are ready
- **Independent agents**: Each branch node reads the shared state but only writes its own key, so branches don't conflict

### 1.1 🗂️ `ParallelState` — Shared State Schema

The state schema holds the original query plus one output field per parallel branch, along with a field for the final synthesized answer.

In [ ]:
# ============================================================================
# PARALLEL STATE: Shared State Schema for the Fan-Out/Fan-In Graph
# ============================================================================
class ParallelState(TypedDict):
    query: str
    research_result: str
    creative_result: str
    technical_result: str
    final_synthesis: str

### 1.2 🧑‍🔬 Building the Parallel Research Graph

`create_parallel_research()` wires up three independent agent nodes (`research`, `creative`, `technical`) that all branch out from `START`, then fan back in to a `synthesize` node that combines their perspectives into one response.

In [ ]:
# ============================================================================
# CREATE_PARALLEL_RESEARCH: Fan-Out Graph Construction
# ============================================================================
def create_parallel_research():
    """Three research agents working in parallel."""

    def research_agent(state: ParallelState) -> dict:
        """Academic/factual research."""
        response = llm.invoke(
            [
                SystemMessage(
                    content="You are an academic researcher. Provide factual, well-sourced information."
                ),
                HumanMessage(content=f"Research this topic: {state['query']}"),
            ]
        )
        return {"research_result": response.content}

    def creative_agent(state: ParallelState) -> dict:
        """Creative perspectives."""
        response = llm.invoke(
            [
                SystemMessage(
                    content="You are a creative thinker. Provide novel perspectives and ideas."
                ),
                HumanMessage(content=f"Give creative insights on: {state['query']}"),
            ]
        )
        return {"creative_result": response.content}

    def technical_agent(state: ParallelState) -> dict:
        """Technical analysis."""
        response = llm.invoke(
            [
                SystemMessage(
                    content="You are a technical analyst. Provide practical, implementation-focused insights."
                ),
                HumanMessage(content=f"Analyze technically: {state['query']}"),
            ]
        )
        return {"technical_result": response.content}

    def synthesize(state: ParallelState) -> dict:
        """Combine all perspectives."""
        synthesis_prompt = f"""Synthesize these three perspectives into a comprehensive response:
        RESEARCH: {state['research_result']}
        CREATIVE: {state['creative_result']}
        TECHNICAL: {state['technical_result']}
        Create a unified, well-structured response."""
        response = llm.invoke(
            [
                SystemMessage(
                    content="You are an expert synthesizer. Combine multiple perspectives into coherent insights."
                ),
                HumanMessage(content=synthesis_prompt),
            ]
        )
        return {"final_synthesis": response.content}

    graph = StateGraph(ParallelState)
    graph.add_node("research", research_agent)
    graph.add_node("creative", creative_agent)
    graph.add_node("technical", technical_agent)
    graph.add_node("synthesize", synthesize)

    # Fan-out: START goes to all three agents
    graph.add_edge(START, "research")
    graph.add_edge(START, "creative")
    graph.add_edge(START, "technical")

    # Fan-in: all three agents feed into synthesize
    graph.add_edge("research", "synthesize")
    graph.add_edge("creative", "synthesize")
    graph.add_edge("technical", "synthesize")
    graph.add_edge("synthesize", END)

    return graph.compile()

### 1.3 ▶️ Running the Parallel Research Demo

`demo_parallel_execution()` compiles the graph, runs it against a sample query, and prints each branch's result alongside the final synthesized answer.

In [ ]:
# ============================================================================
# DEMO_PARALLEL_EXECUTION: Run the Parallel Research Demo
# ============================================================================
def demo_parallel_execution():
    """Demo parallel agent execution."""
    agent = create_parallel_research()

    print("Parallel Agent Execution Demo:\n")
    result = agent.invoke(
        {
            "query": "The future of remote work",
            "research_result": "",
            "creative_result": "",
            "technical_result": "",
            "final_synthesis": "",
        }
    )

    print("Individual Perspectives:")
    print(f"\n[Research]\n{result['research_result'][:300]}...")
    print(f"\n[Creative]\n{result['creative_result'][:300]}...")
    print(f"\n[Technical]\n{result['technical_result'][:300]}...")
    print(f"\n{'='*50}")
    print(f"[SYNTHESIZED]\n{result['final_synthesis']}")

---
## 🗺️ Part 2: Map-Reduce — Parallel Document Summarization

The **map-reduce** pattern applies the same operation independently to each item in a collection ("map") and then combines the individual results into one output ("reduce"). Here, each document is summarized independently and the summaries are then merged into a single overview.

### 2.1 🗂️ `MapReduceState` — Shared State Schema

The state holds the input `documents`, the per-document `summaries` produced by the map step, and the `final_summary` produced by the reduce step.

In [ ]:
# ============================================================================
# MAP_REDUCE_STATE: Shared State Schema for the Map-Reduce Graph
# ============================================================================
class MapReduceState(TypedDict):
    documents: list[str]
    summaries: list[str]
    final_summary: str

### 2.2 🧩 Building the Map-Reduce Summarizer Graph

`create_map_reduce_summarizer()` defines a two-node graph: `map` summarizes every document, and `reduce` combines those summaries into one coherent overview.

In [ ]:
# ============================================================================
# CREATE_MAP_REDUCE_SUMMARIZER: Map-Reduce Graph Construction
# ============================================================================
def create_map_reduce_summarizer():
    """Summarize multiple documents in parallel."""

    def map_summarize(state: MapReduceState) -> dict:
        """Summarize each document (runs in parallel for each)."""
        summaries = []
        for doc in state["documents"]:
            response = llm.invoke(
                [
                    SystemMessage(content="Summarize this document in 2-3 sentences."),
                    HumanMessage(content=doc),
                ]
            )
            summaries.append(response.content)
        return {"summaries": summaries}

    def reduce_combine(state: MapReduceState) -> dict:
        """Combine all summaries."""
        all_summaries = "\n\n".join(
            [f"Summary {i+1}: {s}" for i, s in enumerate(state["summaries"])]
        )
        response = llm.invoke(
            [
                SystemMessage(
                    content="Combine these summaries into one coherent overview."
                ),
                HumanMessage(content=all_summaries),
            ]
        )
        return {"final_summary": response.content}

    graph = StateGraph(MapReduceState)
    graph.add_node("map", map_summarize)
    graph.add_node("reduce", reduce_combine)

    graph.add_edge(START, "map")
    graph.add_edge("map", "reduce")
    graph.add_edge("reduce", END)

    return graph.compile()

### 2.3 ▶️ Running the Map-Reduce Demo

`demo_map_reduce()` runs the graph against three sample documents and prints each individual summary followed by the combined overview.

In [ ]:
# ============================================================================
# DEMO_MAP_REDUCE: Run the Map-Reduce Demo
# ============================================================================
def demo_map_reduce():
    """Demo map-reduce pattern."""
    agent = create_map_reduce_summarizer()
    documents = [
        "Python is a high-level programming language known for its simplicity and readability. It supports multiple programming paradigms and has a vast ecosystem of libraries.",
        "Machine learning is a subset of AI that enables systems to learn from data. Common approaches include supervised, unsupervised, and reinforcement learning.",
        "Cloud computing provides on-demand access to computing resources. Major providers include AWS, Azure, and Google Cloud Platform.",
    ]

    print("\nMap-Reduce Summarization Demo:\n")
    result = agent.invoke(
        {"documents": documents, "summaries": [], "final_summary": ""}
    )

    print("Individual summaries:")
    for i, summary in enumerate(result["summaries"]):
        print(f"  {i+1}. {summary}")
    print(f"\nCombined summary:\n{result['final_summary']}")

---
## 🚀 Part 3: Execute the Demos

The cell below preserves the original script's `__main__` guard verbatim. Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is when executed. Uncomment a line to run that demo instead of (or in addition to) the default.

In [ ]:
# ============================================================================
# RUN: Execute Selected Demo
# ============================================================================
if __name__ == "__main__":
    # demo_parallel_execution()
    demo_map_reduce()

---
## 📝 Summary

In this notebook, we learned:

### 1. Fan-Out/Fan-In Pattern
- **Fan-out**: Multiple `add_edge(START, node)` calls dispatch execution to several agent nodes at once
- **Fan-in**: Multiple upstream nodes can point to the same downstream node, which runs once all of its inputs are available
- **Independent agents**: `research_agent`, `creative_agent`, and `technical_agent` each write to their own state key so they don't interfere with one another, and `synthesize` merges their outputs

### 2. Map-Reduce Pattern
- **Map step**: `map_summarize` applies the same summarization operation independently to every document
- **Reduce step**: `reduce_combine` merges the per-document summaries into a single coherent overview
- This pattern generalizes to any "do the same thing to many items, then combine" workflow

### Next Steps
- Explore how LangGraph's `Send` API can fan out to a *dynamic* number of parallel branches (useful when the number of items isn't known ahead of time)
- Move on to the next notebook in **04 Multi Agent Systems** to see how agents can be orchestrated sequentially or with a supervisor